<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/llm/Flash-Attention/flash-attention.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("flash-attention", ...)`


## Flash Attention Forward Pass

FlashAttention computes exact softmax attention without ever materializing the `N_q × N_k` score matrix. It walks over K/V in tiles and keeps, for every query row, a running max `m`, a running softmax denominator `l` and an unnormalized output accumulator `acc`. For each K/V tile `j`, with scaled scores `S_j = Q K_jᵀ / sqrt(D)`:

```
m_new = max(m, rowmax(S_j))
P_j   = exp(S_j - m_new)
alpha = exp(m - m_new)              # rescales everything accumulated under the old max
l     = alpha * l   + rowsum(P_j)
acc   = alpha * acc + P_j @ V_j
m     = m_new
```

After the last tile, `O = acc / l` and `L = m + log(l)` (the logsumexp of each row of scaled scores, which the backward pass needs). Start from `m = -inf`, `l = 0`, `acc = 0`.

### Part 1 — PyTorch (graded, runs on CPU)

Implement `flash_attention_pytorch(Q, K, V, block_size_q=16, block_size_k=16)`:

- `Q`: `(B, N_q, D)`, `K`, `V`: `(B, N_k, D)`; return `(O, L)` with `O`: `(B, N_q, D)` and `L`: `(B, N_q)`.
- Loop over query tiles and K/V tiles with the recurrence above. Do not compute the full score matrix or call `softmax`.
- The result must equal `F.scaled_dot_product_attention(Q, K, V)` for **any** tile sizes.

### Part 2 — Triton forward kernel (needs a GPU, or the CPU interpreter)

Implement the same loop as a **Triton kernel** `flash_fwd_kernel` that writes `O` and `L`:

- Launch it on a grid of **`(T_q, batch_size)`**: each program instance handles **one tile of `Q`** for a **single batch index**.
- Within each program instance:
  - load the query tile, then stream the batch's K/V tiles through it,
  - apply the online-softmax recurrence above,
  - store the output tile in `O` and the logsumexp values in `L`.
- Address tiles with plain pointer arithmetic (`base + offsets[:, None] * stride_row + offsets[None, :] * stride_col`). `tl.make_block_ptr` / `tl.advance` were removed in Triton 3.9.
- Without a GPU you can run the kernel on CPU with `TRITON_INTERPRET=1` set before importing Triton (slow but exact).

### Notes

- Part 1 is tested with lengths that are **not** multiples of the tile sizes (e.g. `N_q = 40` with 16-row tiles), so the last tile can be partial; slicing `Q[:, q0:q0 + block_size_q]` handles that for you.
- The Triton kernel is tested with sequence lengths that are multiples of the block sizes (powers of 2, at least 16), so it does not need out-of-bounds masks. `D` must be a power of 2 (`tl.arange`).

In [ ]:
import math
import os

import torch
import torch.nn.functional as F

# Triton kernels need a GPU, or Triton's CPU interpreter. For the interpreter, uncomment
# the next line; it must run before triton is first imported in this Python session.
# os.environ["TRITON_INTERPRET"] = "1"
CUDA_AVAILABLE = torch.cuda.is_available()
TRITON_INTERPRET = os.environ.get("TRITON_INTERPRET") == "1"
try:  # guarded so Part 1 still runs on a machine without Triton
    import triton
    import triton.language as tl
    TRITON_AVAILABLE = True
except ImportError:
    TRITON_AVAILABLE = False

print(f"CUDA available: {CUDA_AVAILABLE} | Triton available: {TRITON_AVAILABLE} | "
      f"interpreter: {TRITON_INTERPRET}")

## Part 1 — PyTorch (graded, runs on CPU)


In [ ]:
def flash_attention_pytorch(Q, K, V, block_size_q=16, block_size_k=16):
    """Tiled ("flash") attention forward pass in pure PyTorch.

    Args:
        Q: (B, N_q, D);  K, V: (B, N_k, D)
        block_size_q, block_size_k: tile sizes (N_q, N_k need not be multiples of them)

    Returns:
        O: (B, N_q, D), equal to softmax(Q @ K^T / sqrt(D)) @ V
        L: (B, N_q), the logsumexp of each row of the scaled scores
    """
    # TODO: for every query tile, walk the K/V tiles keeping a running max m, a running
    # denominator l and an unnormalized accumulator acc per query row (the recurrence
    # above); normalize once after the last K/V tile and fill O and L.
    # Do not build the full (N_q, N_k) score matrix and do not call softmax.
    raise NotImplementedError("flash_attention_pytorch")

In [ ]:
# Part 1 tests: exact attention for any tile size, and L = logsumexp of the scaled scores
torch.manual_seed(0)
B_, Nq_, Nk_, D_ = 2, 40, 48, 16       # 40 is not a multiple of 16: the last query tile is partial
Q_, K_, V_ = torch.randn(B_, Nq_, D_), torch.randn(B_, Nk_, D_), torch.randn(B_, Nk_, D_)

O_, L_ = flash_attention_pytorch(Q_, K_, V_)
assert O_.shape == (B_, Nq_, D_) and L_.shape == (B_, Nq_)
assert torch.allclose(O_, F.scaled_dot_product_attention(Q_, K_, V_), atol=1e-5), \
    "O does not match exact attention"
L_ref = torch.logsumexp(Q_ @ K_.transpose(-2, -1) / math.sqrt(D_), dim=-1)
assert torch.allclose(L_, L_ref, atol=1e-5), "L is not the logsumexp of the scaled scores"

# A correct online softmax gives the same answer whatever tiles it walks
for bq, bk in ((8, 8), (16, 48), (40, 5)):
    O2, _ = flash_attention_pytorch(Q_, K_, V_, block_size_q=bq, block_size_k=bk)
    assert torch.allclose(O2, O_, atol=1e-5), f"tile sizes ({bq}, {bk}) changed the answer"

# Large scores must not overflow (that is what the running max is for)
O3, _ = flash_attention_pytorch(Q_ * 30, K_ * 30, V_)
assert torch.isfinite(O3).all(), "inf/NaN on large scores - subtract the running max"
assert torch.allclose(O3, F.scaled_dot_product_attention(Q_ * 30, K_ * 30, V_), atol=1e-4)
print("Part 1 tests passed")

## Part 2 — Triton kernel

Same recurrence, one program per (query tile, batch index). Runs on a CUDA GPU, or on CPU with `TRITON_INTERPRET=1`.


In [ ]:
if TRITON_AVAILABLE:
    @triton.jit
    def flash_fwd_kernel(Q_ptr, K_ptr, V_ptr,
                         O_ptr, L_ptr,
                         stride_qb, stride_qq, stride_qd,
                         stride_kb, stride_kk, stride_kd,
                         stride_vb, stride_vk, stride_vd,
                         stride_ob, stride_ok, stride_od,
                         stride_lb, stride_lq,
                         N_q, N_k,
                         scale,
                         D: tl.constexpr,
                         BLOCK_SIZE_Q: tl.constexpr,
                         BLOCK_SIZE_K: tl.constexpr):

        # Program indices: one query tile of one batch element
        query_tile_index = tl.program_id(0)
        batch_index = tl.program_id(1)

        # Row / column offsets of the tiles this program touches
        offs_q = query_tile_index * BLOCK_SIZE_Q + tl.arange(0, BLOCK_SIZE_Q)
        offs_k = tl.arange(0, BLOCK_SIZE_K)
        offs_d = tl.arange(0, D)

        # (BLOCK_SIZE_Q, D) query tile
        q_ptrs = (Q_ptr + batch_index * stride_qb
                  + offs_q[:, None] * stride_qq + offs_d[None, :] * stride_qd)

        # TODO: pointers to the K tile, read transposed as (D, BLOCK_SIZE_K) so that
        #       tl.dot(q, k) gives the (BLOCK_SIZE_Q, BLOCK_SIZE_K) scores, and to the
        #       (BLOCK_SIZE_K, D) V tile
        # TODO: load q (as float32); m_i = -inf, l_i = 0, acc = 0 (float32)
        # TODO: for start_k in range(0, N_k, BLOCK_SIZE_K): load the K/V tile and apply
        #       the online-softmax recurrence (tl.dot, tl.max, tl.exp, tl.sum)
        # TODO: store acc / l_i to O and m_i + log(l_i) to L
        pass

In [ ]:
# Part 2 test: runs on a CUDA GPU, or on CPU when TRITON_INTERPRET=1 was set before importing triton.
if TRITON_AVAILABLE and (CUDA_AVAILABLE or TRITON_INTERPRET):
    device = "cuda" if CUDA_AVAILABLE else "cpu"
    dtype = torch.float16 if CUDA_AVAILABLE else torch.float32
    B, N_q, N_k, D = 1, 64, 128, 64
    BLOCK_SIZE_Q = 16
    BLOCK_SIZE_K = 16

    torch.manual_seed(0)
    Q = torch.randn((B, N_q, D), dtype=dtype, device=device)
    K = torch.randn((B, N_k, D), dtype=dtype, device=device)
    V = torch.randn((B, N_k, D), dtype=dtype, device=device)

    O = torch.empty((B, N_q, D), dtype=dtype, device=device)
    L = torch.empty((B, N_q), dtype=torch.float32, device=device)

    grid = (triton.cdiv(N_q, BLOCK_SIZE_Q), B)

    flash_fwd_kernel[grid](
        Q, K, V, O, L,
        *Q.stride(), *K.stride(), *V.stride(), *O.stride(), *L.stride(),
        N_q, N_k,
        scale=1.0 / math.sqrt(D),
        D=D,
        BLOCK_SIZE_Q=BLOCK_SIZE_Q,
        BLOCK_SIZE_K=BLOCK_SIZE_K,
    )

    scale = 1.0 / math.sqrt(D)
    scores = torch.matmul(Q.float(), K.float().transpose(-2, -1)) * scale
    O_ref = torch.matmul(torch.softmax(scores, dim=-1), V.float())
    L_ref = torch.logsumexp(scores, dim=-1)

    assert torch.allclose(O.float(), O_ref, atol=2e-2, rtol=1e-2), \
        f"O mismatch (max diff {(O.float() - O_ref).abs().max():.2e})"
    assert torch.allclose(L.float(), L_ref, atol=1e-2, rtol=1e-2), \
        f"L mismatch (max diff {(L.float() - L_ref).abs().max():.2e})"
    print("Triton kernel: O and L match")
else:
    print("Skipping the Triton kernel test - it needs Triton plus a CUDA GPU or TRITON_INTERPRET=1.")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
# Part 1 is graded everywhere. The Triton kernel is graded too when a GPU (or
# TRITON_INTERPRET=1) is available, and reported as not verified otherwise.
check("flash-attention", flash_attention_pytorch=flash_attention_pytorch,
      **({"flash_fwd_kernel": flash_fwd_kernel} if TRITON_AVAILABLE else {}))